# BUSI Dataset - Depthwise & Pointwise Convolution Comparison

Breast Ultrasound Image Classification
Classes: Normal | Benign | Malignant

In [1]:
import os
import math
import random
import copy
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

try:
    import kagglehub
except ImportError:
    kagglehub = None

## 1. REPRODUCIBILITY & DEVICE SETUP

In [2]:
RANDOM_SEED = 7
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
NUM_EPOCHS  = 25
LEARN_RATE  = 1e-3
NUM_CLASSES = 3

CLASS_NAMES = ["normal", "benign", "malignant"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

In [3]:
def fix_random_state(seed: int = RANDOM_SEED) -> None:
    """Pin all random generators for reproducible runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # NOTE: keeping benchmark=True for speed; set deterministic=True if exact reproducibility needed
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

In [4]:
fix_random_state()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORKERS = 0 if os.name == "nt" else 2

print(f"Active device  : {DEVICE}")
print(f"DataLoader CPUs: {WORKERS}")

Active device  : cpu
DataLoader CPUs: 2


## 2. DATASET DISCOVERY

In [5]:
def locate_busi_root() -> Path:
    """
    Search for the BUSI dataset in common local paths.
    Falls back to kagglehub download when not found locally.
    """
    search_paths = [
        Path.cwd() / "BUSI" / "Dataset BUSI",
        Path.cwd() / "Dataset BUSI",
        Path.cwd() / "Dataset_BUSI_with_GT",
    ]

    for p in search_paths:
        if p.exists():
            print(f"Dataset found at: {p}")
            return p

    if kagglehub is not None:
        print("Downloading BUSI dataset via kagglehub …")
        dl_root = Path(kagglehub.dataset_download(
            "subhajournal/busi-breast-ultrasound-images-dataset"
        ))
        required = {"benign", "malignant", "normal"}
        for dirpath, subdirs, _ in os.walk(dl_root):
            if required.issubset({d.lower() for d in subdirs}):
                return Path(dirpath)

    raise FileNotFoundError(
        "BUSI dataset not located. Place it under 'Dataset_BUSI_with_GT/' "
        "or install kagglehub and let it download automatically."
    )

In [6]:
dataset_root = locate_busi_root()
print(f"Dataset root   : {dataset_root}")

100%|██████████| 195M/195M [00:01<00:00, 150MB/s]

Extracting files...


Dataset root   : /root/.cache/kagglehub/datasets/subhajournal/busi-breast-ultrasound-images-dataset/versions/1/Dataset_BUSI_with_GT


## 3. BUILD FILE INDEX

In [7]:
def build_file_index(root: Path) -> pd.DataFrame:
    """
    Walk the dataset root and collect every non-mask PNG image
    alongside its integer class label.
    """
    rows = []
    for cls_name in CLASS_NAMES:
        cls_dir = root / cls_name
        if not cls_dir.exists():
            print(f"  [warn] Class folder missing: {cls_dir}")
            continue
        for img_file in cls_dir.rglob("*.png"):
            if "mask" in img_file.stem.lower():
                continue
            rows.append({"filepath": str(img_file), "class_idx": CLASS_TO_IDX[cls_name]})

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No images indexed — check dataset path and folder names.")
    return df

In [8]:
file_index = build_file_index(dataset_root)
print(f"\nTotal images found : {len(file_index)}")
print("Per-class counts:")
for idx, cnt in file_index["class_idx"].value_counts().sort_index().items():
    print(f"  {IDX_TO_CLASS[idx]:12s}: {cnt}")

print(file_index.head())


Total images found : 780
Per-class counts:
  normal      : 133
  benign      : 437
  malignant   : 210
                                            filepath  class_idx
0  /root/.cache/kagglehub/datasets/subhajournal/b...          0
1  /root/.cache/kagglehub/datasets/subhajournal/b...          0
2  /root/.cache/kagglehub/datasets/subhajournal/b...          0
3  /root/.cache/kagglehub/datasets/subhajournal/b...          0
4  /root/.cache/kagglehub/datasets/subhajournal/b...          0


## 4. TRAIN / VAL / TEST SPLIT

In [9]:
_trainval, test_df = train_test_split(
    file_index,
    test_size=0.15,
    stratify=file_index["class_idx"],
    random_state=RANDOM_SEED,
)

train_df, val_df = train_test_split(
    _trainval,
    test_size=0.15,
    stratify=_trainval["class_idx"],
    random_state=RANDOM_SEED,
)

# Reset indices so iloc lookups work cleanly
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

def report_split(tag: str, df: pd.DataFrame) -> None:
    print(f"\n{tag} ({len(df)} samples):")
    for idx, cnt in df["class_idx"].value_counts().sort_index().items():
        print(f"  {IDX_TO_CLASS[idx]:12s}: {cnt}")

report_split("Train", train_df)
report_split("Validation", val_df)
report_split("Test", test_df)


Train (563 samples):
  normal      : 96
  benign      : 315
  malignant   : 152

Validation (100 samples):
  normal      : 17
  benign      : 56
  malignant   : 27

Test (117 samples):
  normal      : 20
  benign      : 66
  malignant   : 31


## 5. IMAGE PIPELINE

In [10]:
def normalise_to_tensor(img_array: np.ndarray) -> torch.Tensor:
    """Convert HxWxC uint8 array to CxHxW float tensor in [-1, 1]."""
    arr = img_array.astype(np.float32) / 255.0   # → [0, 1]
    arr = (arr - 0.5) / 0.5                       # → [-1, 1]
    return torch.from_numpy(np.transpose(arr, (2, 0, 1)))  # CHW

In [11]:
def apply_augmentations(pil_img: Image.Image, size: tuple = IMG_SIZE) -> torch.Tensor:
    """
    Light augmentation for training images:
      - Random horizontal flip
      - Small rotation  (±12 °)
      - Mild brightness jitter (±15 %)
      - Tiny translation (±10 % of image side)
    """
    # Horizontal flip
    if random.random() < 0.5:
        pil_img = pil_img.transpose(Image.FLIP_LEFT_RIGHT)

    # Rotation
    rot_deg = random.uniform(-12, 12)
    pil_img = pil_img.rotate(rot_deg, resample=Image.BILINEAR)

    # Brightness jitter
    brightness_factor = random.uniform(0.85, 1.15)
    pil_img = ImageEnhance.Brightness(pil_img).enhance(brightness_factor)

    # Translation
    dx = random.uniform(-0.10, 0.10) * size[0]
    dy = random.uniform(-0.10, 0.10) * size[1]
    pil_img = pil_img.transform(
        size, Image.AFFINE, (1, 0, dx, 0, 1, dy), resample=Image.BILINEAR
    )

    arr = np.asarray(pil_img.resize(size))
    return normalise_to_tensor(arr)

In [12]:
def basic_transform(pil_img: Image.Image) -> torch.Tensor:
    arr = np.asarray(pil_img.resize(IMG_SIZE))
    return normalise_to_tensor(arr)

In [13]:
class UltrasoundDataset(Dataset):
    """PyTorch Dataset for BUSI ultrasound images."""

    def __init__(self, dataframe: pd.DataFrame, augment: bool = False):
        self.records  = dataframe
        self.augment  = augment

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, pos: int):
        row   = self.records.iloc[pos]
        label = int(row["class_idx"])
        img   = Image.open(row["filepath"]).convert("RGB")

        tensor = apply_augmentations(img) if self.augment else basic_transform(img)
        return tensor, torch.tensor(label, dtype=torch.long)

## 6. DATALOADERS

In [14]:
def make_loader(dataset: Dataset, shuffle: bool = False, sampler=None) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=(shuffle and sampler is None),
        sampler=sampler,
        num_workers=WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )

In [15]:
train_set    = UltrasoundDataset(train_df, augment=False)
train_aug_set = UltrasoundDataset(train_df, augment=True)
val_set      = UltrasoundDataset(val_df,   augment=False)
test_set     = UltrasoundDataset(test_df,  augment=False)

# Weighted sampler to handle class imbalance
def build_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    counts       = df["class_idx"].value_counts().sort_index()
    inv_freq     = 1.0 / counts
    sample_wts   = df["class_idx"].map(inv_freq).values
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_wts),
        num_samples=len(sample_wts),
        replacement=True,
    )

In [16]:
oversampler = build_weighted_sampler(train_df)

train_loader      = make_loader(train_set,     shuffle=True)
train_aug_loader  = make_loader(train_aug_set, shuffle=True)
train_over_loader = make_loader(train_set,     sampler=oversampler)
val_loader        = make_loader(val_set,       shuffle=False)
test_loader       = make_loader(test_set,      shuffle=False)

## 7. MODEL BUILDING BLOCKS

In [17]:
class ConvBnRelu(nn.Module):
    """
    Standard convolution → Batch Norm → ReLU block.
    Used in the baseline model.
    """
    def __init__(self, in_ch: int, out_ch: int,
                 ksize: int = 3, stride: int = 1, pad: int = 1):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, ksize, stride, pad, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.seq(x)

In [18]:
class DWSepConv(nn.Module):
    """
    Depthwise-Separable Convolution block.
    Factorises a standard convolution into:
      1. Depthwise conv  — per-channel spatial filtering
      2. Pointwise conv  — 1×1 cross-channel mixing
    This reduces parameter count and FLOPs significantly.
    """
    def __init__(self, in_ch: int, out_ch: int,
                 ksize: int = 3, stride: int = 1, pad: int = 1):
        super().__init__()
        # Depthwise: groups == in_ch so each channel filtered independently
        self.dw_conv = nn.Conv2d(in_ch, in_ch, ksize, stride, pad,
                                 groups=in_ch, bias=False)
        self.dw_norm = nn.BatchNorm2d(in_ch)

        # Pointwise: 1×1 conv projects to target channel count
        self.pw_conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.pw_norm = nn.BatchNorm2d(out_ch)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Depthwise stage
        x = self.relu(self.dw_norm(self.dw_conv(x)))
        # Pointwise stage
        x = self.relu(self.pw_norm(self.pw_conv(x)))
        return x

## 8. NETWORK ARCHITECTURES

In [19]:
class StandardCNN(nn.Module):
    """
    Reference CNN using only standard convolutions.
    Serves as the performance baseline.
    Architecture: 4 conv stages → Global Average Pool → FC head
    """
    def __init__(self, num_cls: int = NUM_CLASSES):
        super().__init__()
        self.encoder = nn.Sequential(
            # Stage 1 — 224×224 → 112×112
            ConvBnRelu(3,   32),
            ConvBnRelu(32,  32),
            nn.MaxPool2d(kernel_size=2),

            # Stage 2 — 112×112 → 56×56
            ConvBnRelu(32,  64),
            ConvBnRelu(64,  64),
            nn.MaxPool2d(kernel_size=2),

            # Stage 3 — 56×56 → 28×28
            ConvBnRelu(64,  128),
            ConvBnRelu(128, 128),
            nn.MaxPool2d(kernel_size=2),

            # Stage 4 — compress to 1×1
            ConvBnRelu(128, 256),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_cls),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))

In [20]:
class LightweightDSCNN(nn.Module):
    """
    Lightweight CNN using Depthwise-Separable convolutions throughout.
    Same macro-architecture as StandardCNN but ~8-9× fewer parameters
    in the conv layers.
    """
    def __init__(self, num_cls: int = NUM_CLASSES):
        super().__init__()
        self.encoder = nn.Sequential(
            # Stage 1
            DWSepConv(3,   32),
            DWSepConv(32,  32),
            nn.MaxPool2d(kernel_size=2),

            # Stage 2
            DWSepConv(32,  64),
            DWSepConv(64,  64),
            nn.MaxPool2d(kernel_size=2),

            # Stage 3
            DWSepConv(64,  128),
            DWSepConv(128, 128),
            nn.MaxPool2d(kernel_size=2),

            # Stage 4
            DWSepConv(128, 256),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_cls),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))

In [21]:
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nStandardCNN   params: {count_params(StandardCNN()):,}")
print(f"LightweightDSCNN params: {count_params(LightweightDSCNN()):,}")


StandardCNN   params: 616,163
LightweightDSCNN params: 104,260


## 9. FOCAL LOSS (handles class imbalance)

In [22]:
class FocalCrossEntropy(nn.Module):
    """
    Focal Loss (Lin et al., 2017).
    Down-weights easy examples so the model focuses on hard ones.
    FL(p_t) = -(1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_vals = F.cross_entropy(logits, targets, reduction="none")
        confidence = torch.exp(-ce_vals)                          # p_t
        focal_weight = (1.0 - confidence) ** self.gamma
        loss = focal_weight * ce_vals
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss

## 10. TRAINING LOOP

In [23]:
def run_training(
    model: nn.Module,
    loader_train: DataLoader,
    loader_val: DataLoader,
    epochs: int = NUM_EPOCHS,
    lr: float = LEARN_RATE,
    loss_fn: nn.Module = None,
) -> nn.Module:
    """
    Full training routine with:
      - AdamW optimiser + L2 regularisation
      - ReduceLROnPlateau scheduler (tracks val accuracy)
      - Best-checkpoint tracking (restored at end)
    """
    model = model.to(DEVICE)
    loss_fn = loss_fn or nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="max", factor=0.5, patience=3, verbose=False
    )

    best_val_acc = 0.0
    saved_weights = copy.deepcopy(model.state_dict())

    for ep in range(1, epochs + 1):
        # ── Training phase ──────────────────────
        model.train()
        epoch_loss, n_correct, n_total = 0.0, 0, 0

        for imgs, lbls in loader_train:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)

            opt.zero_grad()
            preds = model(imgs)
            loss  = loss_fn(preds, lbls)
            loss.backward()
            opt.step()

            epoch_loss += loss.item()
            n_correct  += (preds.argmax(dim=1) == lbls).sum().item()
            n_total    += lbls.size(0)

        train_acc  = n_correct / n_total
        mean_loss  = epoch_loss / len(loader_train)

        # ── Validation phase ────────────────────
        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for imgs, lbls in loader_val:
                imgs = imgs.to(DEVICE)
                out  = model(imgs)
                v_correct += (out.argmax(dim=1) == lbls.to(DEVICE)).sum().item()
                v_total   += lbls.size(0)

        val_acc = v_correct / v_total
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc  = val_acc
            saved_weights = copy.deepcopy(model.state_dict())

        print(
            f"  Ep {ep:02d}/{epochs} | "
            f"loss={mean_loss:.4f}  train_acc={train_acc:.4f}  val_acc={val_acc:.4f}"
        )

    model.load_state_dict(saved_weights)
    print(f"  → Best val acc: {best_val_acc:.4f}")
    return model

## 11. EVALUATION UTILITY

In [24]:
def evaluate_on_test(
    model: nn.Module,
    loader: DataLoader,
    display_names: tuple = tuple(n.capitalize() for n in CLASS_NAMES),
) -> tuple:
    """
    Run inference on `loader` and print:
      - Per-class classification report
      - Confusion matrix
    Returns (y_true, y_pred) as Python lists.
    """
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for imgs, lbls in loader:
            out = model(imgs.to(DEVICE))
            all_pred.extend(out.argmax(dim=1).cpu().tolist())
            all_true.extend(lbls.tolist())

    print(classification_report(
        all_true, all_pred,
        target_names=list(display_names),
        zero_division=0,
    ))
    print("Confusion matrix:")
    print(confusion_matrix(all_true, all_pred))
    return all_true, all_pred

## 12. EXPERIMENT RUNS

In [25]:
experiment_log: dict = {}

# ── Exp 1: Baseline Standard CNN ────────────
print("\n" + "=" * 50)
print(" EXP 1 — Standard CNN (Baseline)")
print("=" * 50)
std_cnn = StandardCNN()
std_cnn = run_training(std_cnn, train_loader, val_loader)
print("\n[Test Results]")
gt1, pr1 = evaluate_on_test(std_cnn, test_loader)
experiment_log["Standard CNN"] = (gt1, pr1)

# ── Exp 2: DS-CNN (no tricks) ───────────────
print("\n" + "=" * 50)
print(" EXP 2 — Depthwise-Separable CNN")
print("=" * 50)
ds_cnn = LightweightDSCNN()
ds_cnn = run_training(ds_cnn, train_loader, val_loader)
print("\n[Test Results]")
gt2, pr2 = evaluate_on_test(ds_cnn, test_loader)
experiment_log["DS-CNN"] = (gt2, pr2)

# ── Exp 3: DS-CNN + Oversampling ────────────
print("\n" + "=" * 50)
print(" EXP 3 — DS-CNN + Weighted Oversampling")
print("=" * 50)
ds_over = LightweightDSCNN()
ds_over = run_training(ds_over, train_over_loader, val_loader)
print("\n[Test Results]")
gt3, pr3 = evaluate_on_test(ds_over, test_loader)
experiment_log["DS-CNN + Oversampling"] = (gt3, pr3)

# ── Exp 4: DS-CNN + Augmentation ────────────
print("\n" + "=" * 50)
print(" EXP 4 — DS-CNN + Augmentation")
print("=" * 50)
ds_aug = LightweightDSCNN()
ds_aug = run_training(ds_aug, train_aug_loader, val_loader)
print("\n[Test Results]")
gt4, pr4 = evaluate_on_test(ds_aug, test_loader)
experiment_log["DS-CNN + Augmentation"] = (gt4, pr4)

# ── Exp 5: DS-CNN + Focal Loss ──────────────
print("\n" + "=" * 50)
print(" EXP 5 — DS-CNN + Focal Loss (γ=2)")
print("=" * 50)
ds_focal = LightweightDSCNN()
ds_focal = run_training(
    ds_focal, train_loader, val_loader,
    loss_fn=FocalCrossEntropy(gamma=2.0)
)
print("\n[Test Results]")
gt5, pr5 = evaluate_on_test(ds_focal, test_loader)
experiment_log["DS-CNN + Focal Loss"] = (gt5, pr5)


 EXP 1 — Standard CNN (Baseline)


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

## 13. RESULTS SUMMARY TABLE

In [ ]:
summary_rows = []
for exp_name, (y_true, y_pred) in experiment_log.items():
    summary_rows.append({
        "Experiment"  : exp_name,
        "Accuracy"    : round(accuracy_score(y_true, y_pred), 4),
        "F1 (weighted)": round(f1_score(y_true, y_pred, average="weighted"), 4),
    })

summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values("F1 (weighted)", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 50)
print(" FINAL COMPARISON")
print("=" * 50)
print(summary_df.to_string(index=False))